In [1]:
import sys
print(f"Python version: {sys.version}")
print(f"Executable: {sys.executable}")

Python version: 3.12.9 (tags/v3.12.9:fdb8142, Feb  4 2025, 15:27:58) [MSC v.1942 64 bit (AMD64)]
Executable: d:\AI Projects\agentic-rag-system\.venv\Scripts\python.exe


In [2]:
import os
import sys
from pathlib import Path

# Go one level up from notebooks/ to project root
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")
print(f"Root exists: {ROOT.exists()}")

Project root: D:\AI Projects\agentic-rag-system
Root exists: True


In [3]:
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

GROQ_API_KEY     = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY   = os.getenv("TAVILY_API_KEY")
EMBEDDING_MODEL  = os.getenv("EMBEDDING_MODEL", "BAAI/bge-small-en-v1.5")
SQLITE_DB_PATH   = os.getenv("SQLITE_DB_PATH", "data/processed/memory.db")
FAISS_INDEX_PATH = os.getenv("FAISS_INDEX_PATH", "data/processed/faiss_index")

assert GROQ_API_KEY,   "GROQ_API_KEY missing from .env"
assert TAVILY_API_KEY, "TAVILY_API_KEY missing from .env"

print(f"GROQ_API_KEY     : {GROQ_API_KEY[:8]}...")
print(f"TAVILY_API_KEY   : {TAVILY_API_KEY[:8]}...")
print(f"EMBEDDING_MODEL  : {EMBEDDING_MODEL}")
print(f"SQLITE_DB_PATH   : {SQLITE_DB_PATH}")
print(f"FAISS_INDEX_PATH : {FAISS_INDEX_PATH}")
print("✅ Environment loaded")

python-dotenv could not parse statement starting at line 1


GROQ_API_KEY     : gsk_1EXV...
TAVILY_API_KEY   : tvly-dev...
EMBEDDING_MODEL  : BAAI/bge-small-en-v1.5
SQLITE_DB_PATH   : data/processed/memory.db
FAISS_INDEX_PATH : data/processed/faiss_index
✅ Environment loaded


In [4]:
import logging

# Create logs directory if not exists
log_dir = ROOT / "logs"
log_dir.mkdir(exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler(ROOT / "logs" / "app.log")
    ]
)
logger = logging.getLogger("agentic_rag")
logger.info("✅ Logging initialized")
print("✅ Logger ready — check logs/app.log")

2026-03-17 16:45:37,606 | INFO | agentic_rag | ✅ Logging initialized
✅ Logger ready — check logs/app.log


--- Logging error ---
Traceback (most recent call last):
  File "C:\Users\Ashutosh\AppData\Local\Programs\Python\Python312\Lib\logging\__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
  File "C:\Users\Ashutosh\AppData\Local\Programs\Python\Python312\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2705' in position 47: character maps to <undefined>
Call stack:
  File "C:\Users\Ashutosh\AppData\Local\Programs\Python\Python312\Lib\runpy.py", line 198, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Ashutosh\AppData\Local\Programs\Python\Python312\Lib\runpy.py", line 88, in _run_code
    exec(code, run_globals)
  File "d:\AI Projects\agentic-rag-system\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_

In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY,
    temperature=0.1,
    max_tokens=1024,
)

# Test LLM
response = llm.invoke("Say exactly: Groq LLM initialized successfully")
logger.info(f"LLM response: {response.content}")
print(f"✅ LLM Response: {response.content}")

2026-03-17 16:46:23,263 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 16:46:23,272 | INFO | agentic_rag | LLM response: Groq LLM initialized successfully
✅ LLM Response: Groq LLM initialized successfully


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

# Test embedding
test_vec = embeddings.embed_query("What is RAG?")
logger.info(f"Embedding dim: {len(test_vec)}")
print(f"✅ Embedding model : {EMBEDDING_MODEL}")
print(f"✅ Embedding dim   : {len(test_vec)}")
print(f"✅ Sample values   : {test_vec[:5]}")

2026-03-17 16:46:36,883 | INFO | sentence_transformers.SentenceTransformer | Load pretrained SentenceTransformer: BAAI/bge-small-en-v1.5
2026-03-17 16:46:41,687 | INFO | agentic_rag | Embedding dim: 384
✅ Embedding model : BAAI/bge-small-en-v1.5
✅ Embedding dim   : 384
✅ Sample values   : [-0.03826790302991867, 0.0008686098735779524, 0.02146630734205246, -0.038587331771850586, 0.03674803674221039]


In [7]:
from langchain.schema import Document

raw_docs = [
    Document(
        page_content=(
            "Retrieval-Augmented Generation (RAG) is a technique that enhances "
            "large language models by retrieving relevant documents from an external "
            "knowledge base before generating a response. RAG combines the strengths "
            "of retrieval-based and generation-based approaches. The retrieval step "
            "fetches relevant context, and the generation step uses that context to "
            "produce accurate, grounded answers. RAG reduces hallucinations and keeps "
            "knowledge up-to-date without retraining the model."
        ),
        metadata={"source": "rag_overview.txt", "topic": "RAG"}
    ),
    Document(
        page_content=(
            "FAISS (Facebook AI Similarity Search) is an open-source library for "
            "efficient similarity search and clustering of dense vectors. It is "
            "widely used in RAG systems as a vector store. FAISS supports both "
            "exact and approximate nearest neighbor search. It can handle millions "
            "of vectors efficiently on CPU or GPU. FAISS indexes like IVF and HNSW "
            "enable fast retrieval at scale. It is developed by Meta AI Research."
        ),
        metadata={"source": "faiss_overview.txt", "topic": "VectorDB"}
    ),
    Document(
        page_content=(
            "LangGraph is a library for building stateful multi-actor applications "
            "with LLMs. It extends LangChain with graph-based orchestration. In "
            "LangGraph, nodes represent processing steps and edges define transitions. "
            "It supports cycles, branching, and conditional logic. LangGraph enables "
            "building complex agentic workflows with human-in-the-loop capabilities. "
            "State is persisted across nodes using a shared state object."
        ),
        metadata={"source": "langgraph_overview.txt", "topic": "Framework"}
    ),
    Document(
        page_content=(
            "Groq is an AI inference company that provides extremely fast LLM "
            "inference using custom LPU (Language Processing Unit) hardware. "
            "Groq API supports models like LLaMA3-70B and Mixtral-8x7B. "
            "Groq achieves speeds of 500+ tokens per second making it ideal "
            "for real-time agentic applications. The API is compatible with "
            "OpenAI SDK format. Groq offers a free tier with generous rate limits "
            "suitable for development and prototyping."
        ),
        metadata={"source": "groq_overview.txt", "topic": "LLM"}
    ),
    Document(
        page_content=(
            "Agentic AI systems are AI architectures where LLMs act as autonomous "
            "agents that can reason, plan, use tools, and execute multi-step tasks. "
            "Key components include a router that directs queries, a reasoning agent "
            "that plans steps, tools like web search and code execution, memory for "
            "context persistence, and an evaluation agent for quality control. "
            "Agentic systems use ReAct, Chain-of-Thought, and Tree-of-Thought "
            "for complex reasoning tasks."
        ),
        metadata={"source": "agentic_ai.txt", "topic": "Agents"}
    ),
    Document(
        page_content=(
            "Reranking is a technique used in advanced RAG pipelines to improve "
            "retrieval quality. After initial retrieval a reranker model scores "
            "each retrieved document for relevance to the query. Cross-encoder "
            "models like BGE-Reranker provide more accurate relevance scores than "
            "bi-encoder embeddings. Context compression further reduces noise by "
            "extracting only the most relevant parts of each document. Together "
            "reranking and compression significantly improve answer quality."
        ),
        metadata={"source": "reranking.txt", "topic": "RAG"}
    ),
    Document(
        page_content=(
            "SQLite is a lightweight embedded relational database. In agentic AI "
            "systems SQLite is used for storing conversation history, short-term "
            "memory, long-term memory, and session data. It requires no server "
            "setup and stores everything in a single file. SQLAlchemy provides "
            "a clean ORM interface over SQLite. It is ideal for local development "
            "and small to medium production deployments."
        ),
        metadata={"source": "sqlite_memory.txt", "topic": "Memory"}
    ),
    Document(
        page_content=(
            "Observability in AI systems includes logging, tracing, and metrics. "
            "OpenTelemetry is the standard framework for distributed tracing. "
            "Logging captures individual events. Tracing tracks request flow across "
            "components. Metrics measure system health over time. Together they "
            "enable debugging, performance monitoring, and reliability in production "
            "agentic AI systems."
        ),
        metadata={"source": "observability.txt", "topic": "Observability"}
    ),
]

# Save sample docs to disk
sample_dir = ROOT / "data" / "sample_docs"
sample_dir.mkdir(parents=True, exist_ok=True)
for doc in raw_docs:
    file_path = sample_dir / doc.metadata["source"]
    file_path.write_text(doc.page_content)

logger.info(f"Loaded {len(raw_docs)} documents")
print(f"✅ Loaded {len(raw_docs)} documents")
print(f"✅ Saved to: {sample_dir}")
print(f"✅ Topics covered: {list(set(d.metadata['topic'] for d in raw_docs))}")

2026-03-17 16:46:41,725 | INFO | agentic_rag | Loaded 8 documents
✅ Loaded 8 documents
✅ Saved to: D:\AI Projects\agentic-rag-system\data\sample_docs
✅ Topics covered: ['LLM', 'Observability', 'Framework', 'Memory', 'RAG', 'VectorDB', 'Agents']


In [8]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(raw_docs)

logger.info(f"Created {len(chunks)} chunks from {len(raw_docs)} documents")
print(f"✅ Total chunks     : {len(chunks)}")
print(f"✅ Avg chunk length : {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print(f"\n--- Sample Chunk ---")
print(f"Content  : {chunks[0].page_content[:150]}...")
print(f"Metadata : {chunks[0].metadata}")

2026-03-17 16:46:41,752 | INFO | agentic_rag | Created 17 chunks from 8 documents
✅ Total chunks     : 17
✅ Avg chunk length : 200 chars

--- Sample Chunk ---
Content  : Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by retrieving relevant documents from an external knowledge ba...
Metadata : {'source': 'rag_overview.txt', 'topic': 'RAG'}


In [9]:
from langchain_community.vectorstores import FAISS

# Build FAISS index from chunks
logger.info("Building FAISS index...")

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

# Persist to disk
index_path = ROOT / FAISS_INDEX_PATH
index_path.mkdir(parents=True, exist_ok=True)
vectorstore.save_local(str(index_path))

logger.info(f"FAISS index saved to: {index_path}")
print(f"✅ FAISS index built")
print(f"✅ Total vectors   : {vectorstore.index.ntotal}")
print(f"✅ Saved to        : {index_path}")

2026-03-17 16:46:41,774 | INFO | agentic_rag | Building FAISS index...
2026-03-17 16:46:41,958 | INFO | faiss.loader | Loading faiss with AVX2 support.
2026-03-17 16:46:42,034 | INFO | faiss.loader | Successfully loaded faiss with AVX2 support.
2026-03-17 16:46:42,042 | INFO | faiss | Failed to load GPU Faiss: name 'GpuIndexIVFFlat' is not defined. Will not load constructor refs for GPU indexes. This is only an error if you're trying to use GPU Faiss.
2026-03-17 16:46:42,042 | INFO | agentic_rag | FAISS index saved to: D:\AI Projects\agentic-rag-system\data\processed\faiss_index
✅ FAISS index built
✅ Total vectors   : 17
✅ Saved to        : D:\AI Projects\agentic-rag-system\data\processed\faiss_index


In [10]:
# Test 1 — Basic similarity search
query = "How does RAG reduce hallucinations?"

results = vectorstore.similarity_search_with_score(query, k=3)

print(f"Query: {query}")
print(f"\n--- Top {len(results)} Results ---")
for i, (doc, score) in enumerate(results):
    print(f"\n[{i+1}] Score   : {score:.4f}")
    print(f"     Source  : {doc.metadata.get('source')}")
    print(f"     Topic   : {doc.metadata.get('topic')}")
    print(f"     Content : {doc.page_content[:120]}...")

logger.info(f"Similarity search returned {len(results)} results")

Query: How does RAG reduce hallucinations?

--- Top 3 Results ---

[1] Score   : 0.5999
     Source  : rag_overview.txt
     Topic   : RAG
     Content : . The retrieval step fetches relevant context, and the generation step uses that context to produce accurate, grounded a...

[2] Score   : 0.7835
     Source  : rag_overview.txt
     Topic   : RAG
     Content : Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by retrieving relevant documents...

[3] Score   : 0.9252
     Source  : reranking.txt
     Topic   : RAG
     Content : . Context compression further reduces noise by extracting only the most relevant parts of each document. Together rerank...
2026-03-17 16:46:42,077 | INFO | agentic_rag | Similarity search returned 3 results


In [11]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_community.vectorstores import FAISS

# Base retriever with MMR for diversity
base_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,
        "fetch_k": 8,
        "lambda_mult": 0.7
    }
)

# Compressor — extracts only relevant parts from each doc
compressor = LLMChainExtractor.from_llm(llm)

# Advanced retriever = base retriever + compression
advanced_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

# Test advanced retrieval
test_query = "What is FAISS and why is it used in RAG?"
compressed_docs = advanced_retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"\n--- Compressed Results ({len(compressed_docs)} docs) ---")
for i, doc in enumerate(compressed_docs):
    print(f"\n[{i+1}] Source  : {doc.metadata.get('source')}")
    print(f"     Content : {doc.page_content[:200]}")

logger.info(f"Advanced retrieval returned {len(compressed_docs)} compressed docs")
print(f"\n✅ Advanced RAG retriever ready")

2026-03-17 16:46:42,527 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 16:46:42,622 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 16:46:42,732 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 16:46:43,029 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Query: What is FAISS and why is it used in RAG?

--- Compressed Results (2 docs) ---

[1] Source  : faiss_overview.txt
     Content : FAISS (Facebook AI Similarity Search) is an open-source library for efficient similarity search and clustering of dense vectors. It is widely used in RAG systems as a vector store. FAISS supports both

[2] Source  : faiss_overview.txt
     Content : . It can handle millions of vectors efficiently on CPU or GPU. FAISS indexes like IVF and HNSW enable fast retri

In [12]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

# RAG Prompt
RAG_PROMPT = ChatPromptTemplate.from_template("""
You are an expert AI assistant for an Agentic RAG system.
Answer the question using ONLY the provided context.
If the context does not contain enough information, say so clearly.
Always mention which source you used.

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    """Format retrieved docs into context string."""
    return "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}]\n{doc.page_content}"
        for doc in docs
    )

# Basic RAG chain
basic_rag_chain = (
    {
        "context": base_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# Advanced RAG chain (with compression)
advanced_rag_chain = (
    {
        "context": advanced_retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

# Test both chains
test_questions = [
    "What is RAG and how does it reduce hallucinations?",
    "What makes Groq fast for LLM inference?",
    "How does LangGraph handle state?"
]

print("=== Basic RAG Pipeline ===\n")
for q in test_questions:
    answer = basic_rag_chain.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer[:300]}")
    print("-" * 60)

logger.info("RAG pipeline tested successfully")
print("\n✅ RAG Pipeline ready")

=== Basic RAG Pipeline ===

2026-03-17 16:46:43,585 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Q: What is RAG and how does it reduce hallucinations?
A: RAG (Retrieval-Augmented Generation) is a technique that enhances large language models by retrieving relevant documents from an external knowledge base before generating a response. According to [Source: rag_overview.txt], RAG reduces hallucinations and keeps knowledge up-to-date without retraining
------------------------------------------------------------
2026-03-17 16:46:43,795 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Q: What makes Groq fast for LLM inference?
A: According to the context provided in [Source: groq_overview.txt], Groq is fast for LLM inference due to its custom LPU (Language Processing Unit) hardware.
------------------------------------------------------------
2026-03-17 16:46:44,061 | INFO | httpx

In [13]:
from tavily import TavilyClient

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

def web_search(query: str, max_results: int = 3) -> str:
    """
    Search the web using Tavily API.
    
    Args:
        query: Search query string
        max_results: Number of results to return
    
    Returns:
        Formatted string of search results
    """
    try:
        logger.info(f"Web search query: {query}")
        response = tavily_client.search(
            query=query,
            max_results=max_results,
            search_depth="basic"
        )
        results = response.get("results", [])
        if not results:
            return "No results found."
        
        formatted = []
        for i, r in enumerate(results):
            formatted.append(
                f"[{i+1}] Title   : {r.get('title', 'N/A')}\n"
                f"     URL     : {r.get('url', 'N/A')}\n"
                f"     Content : {r.get('content', 'N/A')[:250]}"
            )
        result_str = "\n\n".join(formatted)
        logger.info(f"Web search returned {len(results)} results")
        return result_str

    except Exception as e:
        logger.error(f"Web search failed: {e}")
        return f"Search failed: {str(e)}"

# Test web search
print("=== Web Search Tool Test ===\n")
result = web_search("latest RAG techniques in AI 2025")
print(result)
print("\n✅ Web Search Tool ready")

=== Web Search Tool Test ===

2026-03-17 16:47:33,349 | INFO | agentic_rag | Web search query: latest RAG techniques in AI 2025
2026-03-17 16:47:35,801 | INFO | agentic_rag | Web search returned 3 results
[1] Title   : RAG Architectures: A Complete Guide for 2025 - Medium
     URL     : https://medium.com/data-science-collective/rag-architectures-a-complete-guide-for-2025-daf98a2ede8c
     Content : Modern RAG systems have moved beyond sparse retrieval methods like TF-IDF or BM25. Instead, they utilize dense, transformer-based embeddings (

[2] Title   : RAG in 2025: 7 Proven Strategies to Deploy Retrieval-Augmented ...
     URL     : https://www.morphik.ai/blog/retrieval-augmented-generation-strategies
     Content : Through hallucination mitigation techniques that ground responses in verified documents, RAG transforms unreliable AI outputs into trustworthy business

[3] Title   : RAG in 2025 The New Evolution of Retrieval Augmented Generation ...
     URL     : https://ai.plainenglis

In [15]:
import io
import contextlib
import traceback

def execute_python(code: str) -> dict:
    """
    Safely execute Python code and capture output.
    
    Args:
        code: Python code string to execute
    
    Returns:
        Dict with success status, stdout, stderr, error
    """
    stdout_capture = io.StringIO()
    stderr_capture = io.StringIO()

    try:
        logger.info("Executing Python code block")
        with contextlib.redirect_stdout(stdout_capture):
            with contextlib.redirect_stderr(stderr_capture):
                exec(code, {"__builtins__": __builtins__})
        return {
            "success": True,
            "stdout": stdout_capture.getvalue(),
            "stderr": stderr_capture.getvalue(),
            "error": None
        }
    except Exception as e:
        logger.error(f"Python execution failed: {e}")
        return {
            "success": False,
            "stdout": stdout_capture.getvalue(),
            "stderr": stderr_capture.getvalue(),
            "error": traceback.format_exc()
        }

# Test 1 — Basic math
print("=== Python Executor Tool Test ===\n")

test1 = """
import math
numbers = [1, 4, 9, 16, 25, 36]
results = [math.sqrt(n) for n in numbers]
print(f"Square roots: {results}")
print(f"Sum: {sum(results):.4f}")
"""
r1 = execute_python(test1)
print(f"Test 1 - Basic Math:")
print(f"  Success: {r1['success']}")
print(f"  Output : {r1['stdout']}")

# Test 2 — Data analysis
test2 = """
data = [10, 20, 30, 40, 50]
mean = sum(data) / len(data)
variance = sum((x - mean)**2 for x in data) / len(data)
print(f"Mean    : {mean}")
print(f"Variance: {variance}")
print(f"Std Dev : {variance**0.5:.4f}")
"""
r2 = execute_python(test2)
print(f"\nTest 2 - Data Analysis:")
print(f"  Success: {r2['success']}")
print(f"  Output : {r2['stdout']}")

# Test 3 — Error handling
test3 = """
result = 10 / 0
"""
r3 = execute_python(test3)
print(f"\nTest 3 - Error Handling:")
print(f"  Success: {r3['success']}")
print(f"  Error  : {r3['error'][:100]}...")

print("\n✅ Python Executor Tool ready")

=== Python Executor Tool Test ===

2026-03-17 16:48:18,581 | INFO | agentic_rag | Executing Python code block
Test 1 - Basic Math:
  Success: True
  Output : Square roots: [1.0, 2.0, 3.0, 4.0, 5.0, 6.0]
Sum: 21.0000

2026-03-17 16:48:18,581 | INFO | agentic_rag | Executing Python code block

Test 2 - Data Analysis:
  Success: True
  Output : Mean    : 30.0
Variance: 200.0
Std Dev : 14.1421

2026-03-17 16:48:18,581 | INFO | agentic_rag | Executing Python code block
2026-03-17 16:48:18,584 | ERROR | agentic_rag | Python execution failed: division by zero

Test 3 - Error Handling:
  Success: False
  Error  : Traceback (most recent call last):
  File "C:\Users\Ashutosh\AppData\Local\Temp\ipykernel_1912\23441...

✅ Python Executor Tool ready


In [16]:
import sqlite3
import json
from datetime import datetime

db_tool_path = ROOT / "data" / "processed" / "tool_results.db"
db_tool_path.parent.mkdir(parents=True, exist_ok=True)

def init_tool_db(path: str) -> sqlite3.Connection:
    """Initialize database for storing tool results."""
    conn = sqlite3.connect(str(path))
    conn.execute("""
        CREATE TABLE IF NOT EXISTS tool_results (
            id        INTEGER PRIMARY KEY AUTOINCREMENT,
            tool_name TEXT NOT NULL,
            query     TEXT NOT NULL,
            result    TEXT NOT NULL,
            success   INTEGER NOT NULL,
            timestamp TEXT NOT NULL
        )
    """)
    conn.commit()
    return conn

def save_tool_result(conn, tool_name: str, query: str, result: str, success: bool):
    """Save a tool execution result to database."""
    conn.execute(
        "INSERT INTO tool_results (tool_name, query, result, success, timestamp) VALUES (?, ?, ?, ?, ?)",
        (tool_name, query, result[:1000], int(success), datetime.now().isoformat())
    )
    conn.commit()
    logger.info(f"Saved tool result: {tool_name}")

def get_tool_results(conn, tool_name: str = None, limit: int = 5) -> list:
    """Retrieve recent tool results from database."""
    if tool_name:
        cursor = conn.execute(
            "SELECT tool_name, query, result, success, timestamp FROM tool_results WHERE tool_name=? ORDER BY id DESC LIMIT ?",
            (tool_name, limit)
        )
    else:
        cursor = conn.execute(
            "SELECT tool_name, query, result, success, timestamp FROM tool_results ORDER BY id DESC LIMIT ?",
            (limit,)
        )
    return cursor.fetchall()

def query_database(conn, sql: str) -> list:
    """Execute a read-only SQL query."""
    try:
        cursor = conn.execute(sql)
        return cursor.fetchall()
    except Exception as e:
        logger.error(f"Database query failed: {e}")
        return []

# Initialize and test
tool_db = init_tool_db(str(db_tool_path))

# Save some test results
save_tool_result(tool_db, "web_search", "RAG techniques", "Found 3 results", True)
save_tool_result(tool_db, "python_executor", "math calculation", "Output: 42", True)
save_tool_result(tool_db, "web_search", "Groq API", "Found 2 results", True)

# Query results
print("=== Database Tool Test ===\n")
all_results = get_tool_results(tool_db, limit=5)
print(f"Stored {len(all_results)} tool results:")
for r in all_results:
    print(f"  Tool: {r[0]} | Query: {r[1][:30]} | Success: {bool(r[3])} | Time: {r[4][:19]}")

# Test raw SQL query
count = query_database(tool_db, "SELECT COUNT(*) FROM tool_results")
print(f"\nTotal records in DB: {count[0][0]}")
print("\n✅ Database Tool ready")

2026-03-17 16:49:03,432 | INFO | agentic_rag | Saved tool result: web_search
2026-03-17 16:49:03,432 | INFO | agentic_rag | Saved tool result: python_executor
2026-03-17 16:49:03,440 | INFO | agentic_rag | Saved tool result: web_search
=== Database Tool Test ===

Stored 3 tool results:
  Tool: web_search | Query: Groq API | Success: True | Time: 2026-03-17T16:49:03
  Tool: python_executor | Query: math calculation | Success: True | Time: 2026-03-17T16:49:03
  Tool: web_search | Query: RAG techniques | Success: True | Time: 2026-03-17T16:49:03

Total records in DB: 3

✅ Database Tool ready


In [17]:
import sqlite3
from datetime import datetime

memory_db_path = ROOT / SQLITE_DB_PATH
memory_db_path.parent.mkdir(parents=True, exist_ok=True)

def init_memory_db(path: str) -> sqlite3.Connection:
    """Initialize SQLite memory database with all tables."""
    conn = sqlite3.connect(str(path))
    
    # Short term memory — current session messages
    conn.execute("""
        CREATE TABLE IF NOT EXISTS short_term_memory (
            id        INTEGER PRIMARY KEY AUTOINCREMENT,
            session   TEXT NOT NULL,
            role      TEXT NOT NULL,
            content   TEXT NOT NULL,
            timestamp TEXT NOT NULL
        )
    """)
    conn.commit()
    logger.info(f"Memory DB initialized at: {path}")
    return conn

def save_short_term(conn, session: str, role: str, content: str):
    """Save message to short term memory."""
    conn.execute(
        "INSERT INTO short_term_memory (session, role, content, timestamp) VALUES (?, ?, ?, ?)",
        (session, role, content, datetime.now().isoformat())
    )
    conn.commit()

def get_short_term(conn, session: str, limit: int = 10) -> list:
    """Get recent messages from short term memory."""
    cursor = conn.execute(
        "SELECT role, content, timestamp FROM short_term_memory WHERE session=? ORDER BY id DESC LIMIT ?",
        (session, limit)
    )
    return list(reversed(cursor.fetchall()))

def clear_short_term(conn, session: str):
    """Clear short term memory for a session."""
    conn.execute("DELETE FROM short_term_memory WHERE session=?", (session,))
    conn.commit()
    logger.info(f"Cleared short term memory for session: {session}")

# Test short term memory
memory_conn = init_memory_db(str(memory_db_path))
session_id = "test_session_001"

# Simulate a conversation
save_short_term(memory_conn, session_id, "user", "What is RAG?")
save_short_term(memory_conn, session_id, "assistant", "RAG stands for Retrieval Augmented Generation...")
save_short_term(memory_conn, session_id, "user", "How does FAISS work?")
save_short_term(memory_conn, session_id, "assistant", "FAISS is a vector similarity search library...")

# Retrieve conversation
history = get_short_term(memory_conn, session_id)
print("=== Short Term Memory Test ===\n")
print(f"Session: {session_id}")
print(f"Messages stored: {len(history)}\n")
for role, content, ts in history:
    print(f"  [{role.upper()}] {content[:60]}...")
    print(f"           Time: {ts[:19]}")

print("\n✅ Short Term Memory ready")

2026-03-17 16:49:32,021 | INFO | agentic_rag | Memory DB initialized at: D:\AI Projects\agentic-rag-system\data\processed\memory.db
=== Short Term Memory Test ===

Session: test_session_001
Messages stored: 4

  [USER] What is RAG?...
           Time: 2026-03-17T16:49:32
  [ASSISTANT] RAG stands for Retrieval Augmented Generation......
           Time: 2026-03-17T16:49:32
  [USER] How does FAISS work?...
           Time: 2026-03-17T16:49:32
  [ASSISTANT] FAISS is a vector similarity search library......
           Time: 2026-03-17T16:49:32

✅ Short Term Memory ready


In [18]:
def init_long_term_table(conn):
    """Add long term memory table to existing connection."""
    conn.execute("""
        CREATE TABLE IF NOT EXISTS long_term_memory (
            id        INTEGER PRIMARY KEY AUTOINCREMENT,
            key       TEXT UNIQUE NOT NULL,
            value     TEXT NOT NULL,
            category  TEXT,
            metadata  TEXT,
            timestamp TEXT NOT NULL
        )
    """)
    conn.commit()

def save_long_term(conn, key: str, value: str, category: str = "general", metadata: dict = None):
    """Save important fact to long term memory."""
    conn.execute(
        "INSERT OR REPLACE INTO long_term_memory (key, value, category, metadata, timestamp) VALUES (?, ?, ?, ?, ?)",
        (key, value, category, json.dumps(metadata or {}), datetime.now().isoformat())
    )
    conn.commit()
    logger.info(f"Saved to long term memory: {key}")

def get_long_term(conn, key: str) -> str:
    """Retrieve a fact from long term memory by key."""
    cursor = conn.execute(
        "SELECT value FROM long_term_memory WHERE key=?", (key,)
    )
    row = cursor.fetchone()
    return row[0] if row else None

def get_long_term_by_category(conn, category: str) -> list:
    """Retrieve all facts in a category."""
    cursor = conn.execute(
        "SELECT key, value, timestamp FROM long_term_memory WHERE category=? ORDER BY timestamp DESC",
        (category,)
    )
    return cursor.fetchall()

def search_long_term(conn, keyword: str) -> list:
    """Search long term memory by keyword."""
    cursor = conn.execute(
        "SELECT key, value, category FROM long_term_memory WHERE value LIKE ? OR key LIKE ?",
        (f"%{keyword}%", f"%{keyword}%")
    )
    return cursor.fetchall()

# Initialize long term table
init_long_term_table(memory_conn)

# Store important facts
save_long_term(memory_conn, "user_expertise", "Advanced Python developer interested in AI", "user_profile")
save_long_term(memory_conn, "preferred_model", "llama-3.3-70b-versatile on Groq", "preferences")
save_long_term(memory_conn, "project_name", "Agentic RAG System", "project")
save_long_term(memory_conn, "embedding_model", "BAAI/bge-small-en-v1.5", "project")
save_long_term(memory_conn, "vector_db", "FAISS with cosine similarity", "project")

# Test retrieval
print("=== Long Term Memory Test ===\n")

val = get_long_term(memory_conn, "user_expertise")
print(f"Key lookup 'user_expertise': {val}")

project_facts = get_long_term_by_category(memory_conn, "project")
print(f"\nAll project facts ({len(project_facts)}):")
for key, value, ts in project_facts:
    print(f"  {key}: {value}")

search_results = search_long_term(memory_conn, "FAISS")
print(f"\nSearch for 'FAISS' ({len(search_results)} results):")
for key, value, category in search_results:
    print(f"  [{category}] {key}: {value[:60]}")

print("\n✅ Long Term Memory ready")

2026-03-17 16:49:56,654 | INFO | agentic_rag | Saved to long term memory: user_expertise
2026-03-17 16:49:56,656 | INFO | agentic_rag | Saved to long term memory: preferred_model
2026-03-17 16:49:56,660 | INFO | agentic_rag | Saved to long term memory: project_name
2026-03-17 16:49:56,662 | INFO | agentic_rag | Saved to long term memory: embedding_model
2026-03-17 16:49:56,662 | INFO | agentic_rag | Saved to long term memory: vector_db
=== Long Term Memory Test ===

Key lookup 'user_expertise': Advanced Python developer interested in AI

All project facts (3):
  vector_db: FAISS with cosine similarity
  embedding_model: BAAI/bge-small-en-v1.5
  project_name: Agentic RAG System

Search for 'FAISS' (1 results):
  [project] vector_db: FAISS with cosine similarity

✅ Long Term Memory ready


In [19]:
class MemoryManager:
    """
    Unified manager for short-term and long-term memory.
    Handles context building for LLM prompts.
    """

    def __init__(self, conn: sqlite3.Connection, session_id: str):
        self.conn = conn
        self.session_id = session_id
        logger.info(f"MemoryManager initialized for session: {session_id}")

    def remember(self, role: str, content: str):
        """Save message to short term memory."""
        save_short_term(self.conn, self.session_id, role, content)

    def recall_recent(self, limit: int = 6) -> list:
        """Get recent conversation history."""
        return get_short_term(self.conn, self.session_id, limit)

    def learn(self, key: str, value: str, category: str = "general"):
        """Save important fact to long term memory."""
        save_long_term(self.conn, key, value, category)

    def recall_fact(self, key: str) -> str:
        """Retrieve a specific fact from long term memory."""
        return get_long_term(self.conn, key)

    def build_context(self, limit: int = 6) -> str:
        """Build conversation context string for LLM prompt."""
        history = self.recall_recent(limit)
        if not history:
            return "No previous conversation."
        lines = []
        for role, content, ts in history:
            lines.append(f"{role.upper()}: {content}")
        return "\n".join(lines)

    def get_user_profile(self) -> dict:
        """Get stored user preferences and profile."""
        return {
            "expertise"  : self.recall_fact("user_expertise"),
            "preference" : self.recall_fact("preferred_model"),
            "project"    : self.recall_fact("project_name"),
        }

    def clear_session(self):
        """Clear current session short term memory."""
        clear_short_term(self.conn, self.session_id)

# Test Memory Manager
print("=== Memory Manager Test ===\n")

manager = MemoryManager(memory_conn, "test_session_002")

manager.remember("user", "Explain the difference between RAG and fine-tuning")
manager.remember("assistant", "RAG retrieves external knowledge at inference time while fine-tuning bakes knowledge into model weights")
manager.remember("user", "Which approach is better for frequently changing data?")
manager.remember("assistant", "RAG is better for frequently changing data since you only update the vector store")

manager.learn("last_topic", "RAG vs fine-tuning comparison", "session_facts")

context = manager.build_context()
print("Built Context:")
print(context)

profile = manager.get_user_profile()
print(f"\nUser Profile: {profile}")

print("\n✅ Memory Manager ready")

=== Memory Manager Test ===

2026-03-17 16:50:39,396 | INFO | agentic_rag | MemoryManager initialized for session: test_session_002
2026-03-17 16:50:39,412 | INFO | agentic_rag | Saved to long term memory: last_topic
Built Context:
USER: Explain the difference between RAG and fine-tuning
ASSISTANT: RAG retrieves external knowledge at inference time while fine-tuning bakes knowledge into model weights
USER: Which approach is better for frequently changing data?
ASSISTANT: RAG is better for frequently changing data since you only update the vector store

User Profile: {'expertise': 'Advanced Python developer interested in AI', 'preference': 'llama-3.3-70b-versatile on Groq', 'project': 'Agentic RAG System'}

✅ Memory Manager ready


In [21]:
import re
import logging

# Logger setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Blocked patterns
BLOCKED_PATTERNS = [
    r"\b(hack|exploit|malware|virus|ransomware)\b",
    r"\b(bomb|weapon|explosive|poison)\b",
    r"\b(suicide|self.harm|kill yourself)\b",
    r"\b(password|credit.card|ssn|social.security)\b",
]

# Max input length
MAX_INPUT_LENGTH = 2000

def check_input_guardrails(query: str) -> tuple:
    """
    Validate and sanitize user input.
    
    Returns:
        (is_safe: bool, reason: str, sanitized_query: str)
    """

    if not query or not query.strip():
        return False, "Empty input provided", ""

    if len(query) > MAX_INPUT_LENGTH:
        return False, f"Input too long: {len(query)} chars (max {MAX_INPUT_LENGTH})", ""

    query_lower = query.lower()

    for pattern in BLOCKED_PATTERNS:
        if re.search(pattern, query_lower):
            logger.warning(f"Blocked input pattern detected: {pattern}")
            return False, "Input contains blocked content", ""

    sanitized = " ".join(query.strip().split())

    logger.info(f"Input guardrail passed: {sanitized[:50]}...")
    return True, "Input is safe", sanitized


# Test
print("=== Input Guardrails Test ===\n")

test_inputs = [
    "What is RAG and how does it work?",
    "",
    "How to hack into a system?",
    "A" * 2500,
    "  What   is   FAISS?  ",
    "Explain agentic AI systems",
]

for inp in test_inputs:
    is_safe, reason, sanitized = check_input_guardrails(inp)
    display = inp[:40] + "..." if len(inp) > 40 else inp
    print(f"Input  : '{display}'")
    print(f"Safe   : {is_safe}")
    print(f"Reason : {reason}")
    if sanitized:
        print(f"Clean  : '{sanitized[:60]}'")
    print("-" * 50)

print("\n✅ Input Guardrails ready")

=== Input Guardrails Test ===

2026-03-17 16:54:45,923 | INFO | __main__ | Input guardrail passed: What is RAG and how does it work?...
Input  : 'What is RAG and how does it work?'
Safe   : True
Reason : Input is safe
Clean  : 'What is RAG and how does it work?'
--------------------------------------------------
Input  : ''
Safe   : False
Reason : Empty input provided
--------------------------------------------------
2026-03-17 16:54:45,923 | WARNING | __main__ | Blocked input pattern detected: \b(hack|exploit|malware|virus|ransomware)\b
Input  : 'How to hack into a system?'
Safe   : False
Reason : Input contains blocked content
--------------------------------------------------
Input  : 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA...'
Safe   : False
Reason : Input too long: 2500 chars (max 2000)
--------------------------------------------------
2026-03-17 16:54:45,923 | INFO | __main__ | Input guardrail passed: What is FAISS?...
Input  : '  What   is   FAISS?  '
Safe   : True
Reason : 

In [22]:
# Blocked output patterns
OUTPUT_BLOCKED_PATTERNS = [
    r"\b(as an AI|I am an AI|I'm an AI)\b",
    r"\b(I cannot|I can't|I am not able)\b",
]

MAX_OUTPUT_LENGTH = 4000

def check_output_guardrails(response: str) -> tuple:
    """
    Validate and clean LLM output before returning to user.
    
    Args:
        response: Raw LLM response string
    
    Returns:
        Tuple of (is_valid: bool, reason: str, cleaned_response: str)
    """
    # Check empty output
    if not response or not response.strip():
        return False, "Empty response from LLM", ""

    # Check minimum length — too short means likely failed
    if len(response.strip()) < 10:
        return False, "Response too short — likely a failure", ""

    # Truncate if too long
    if len(response) > MAX_OUTPUT_LENGTH:
        response = response[:MAX_OUTPUT_LENGTH] + "...[truncated]"
        logger.warning("Output truncated due to length")

    # Remove common LLM refusal patterns
    cleaned = response.strip()
    for pattern in OUTPUT_BLOCKED_PATTERNS:
        cleaned = re.sub(pattern, "", cleaned, flags=re.IGNORECASE)

    # Remove excessive newlines
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)

    # Remove leading/trailing whitespace
    cleaned = cleaned.strip()

    logger.info("Output guardrail passed")
    return True, "Output is valid", cleaned

def validate_pipeline_output(query: str, response: str) -> dict:
    """
    Full validation of input + output pair.
    
    Args:
        query: Original user query
        response: LLM generated response

    Returns:
        Dict with validation results
    """
    input_safe, input_reason, clean_query = check_input_guardrails(query)
    output_valid, output_reason, clean_response = check_output_guardrails(response)

    return {
        "input_safe"     : input_safe,
        "input_reason"   : input_reason,
        "output_valid"   : output_valid,
        "output_reason"  : output_reason,
        "clean_query"    : clean_query,
        "clean_response" : clean_response,
        "pipeline_ok"    : input_safe and output_valid
    }

# Test output guardrails
print("=== Output Guardrails Test ===\n")

test_outputs = [
    "RAG is a technique that combines retrieval with generation to produce accurate answers.",
    "",
    "I cannot answer that question as an AI language model.",
    "RAG " * 1500,
    "  RAG reduces hallucinations by grounding responses in retrieved documents.  ",
]

for out in test_outputs:
    is_valid, reason, cleaned = check_output_guardrails(out)
    display = out[:50] + "..." if len(out) > 50 else out
    print(f"Output : '{display}'")
    print(f"Valid  : {is_valid}")
    print(f"Reason : {reason}")
    if cleaned:
        print(f"Cleaned: '{cleaned[:80]}'")
    print("-" * 50)

# Test full pipeline validation
print("\n=== Full Pipeline Validation ===\n")
result = validate_pipeline_output(
    "What is RAG?",
    "RAG stands for Retrieval Augmented Generation. It enhances LLMs by retrieving relevant context."
)
for k, v in result.items():
    print(f"  {k}: {v}")

print("\n✅ Output Guardrails ready")

=== Output Guardrails Test ===

2026-03-17 16:56:58,247 | INFO | __main__ | Output guardrail passed
Output : 'RAG is a technique that combines retrieval with ge...'
Valid  : True
Reason : Output is valid
Cleaned: 'RAG is a technique that combines retrieval with generation to produce accurate a'
--------------------------------------------------
Output : ''
Valid  : False
Reason : Empty response from LLM
--------------------------------------------------
2026-03-17 16:56:58,247 | INFO | __main__ | Output guardrail passed
Output : 'I cannot answer that question as an AI language mo...'
Valid  : True
Reason : Output is valid
Cleaned: 'answer that question  language model.'
--------------------------------------------------
2026-03-17 16:56:58,247 | WARNING | __main__ | Output truncated due to length
2026-03-17 16:56:58,247 | INFO | __main__ | Output guardrail passed
Output : 'RAG RAG RAG RAG RAG RAG RAG RAG RAG RAG RAG RAG RA...'
Valid  : True
Reason : Output is valid
Cleaned: 'RAG RAG RA

In [23]:
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser

ROUTER_PROMPT = ChatPromptTemplate.from_template("""
You are a routing agent. Your job is to analyze the user query and decide 
which agent should handle it. Choose EXACTLY ONE of these options:

- RAG_AGENT      : Query needs information from the knowledge base
- WEB_SEARCH     : Query needs current/live information from the internet  
- PYTHON_AGENT   : Query needs code execution or mathematical computation
- REASONING_AGENT: Query needs multi-step reasoning or complex analysis
- DIRECT_ANSWER  : Query is simple and can be answered directly

Respond with ONLY the agent name, nothing else.

Query: {query}

Agent:
""")

router_chain = ROUTER_PROMPT | llm | StrOutputParser()

def route_query(query: str) -> str:
    """
    Route a query to the appropriate agent.
    
    Args:
        query: User query string
    
    Returns:
        Agent name string
    """
    try:
        # Check guardrails first
        is_safe, reason, clean_query = check_input_guardrails(query)
        if not is_safe:
            logger.warning(f"Query blocked by guardrails: {reason}")
            return "BLOCKED"

        route = router_chain.invoke({"query": clean_query}).strip().upper()

        # Validate route is one of expected values
        valid_routes = ["RAG_AGENT", "WEB_SEARCH", "PYTHON_AGENT", "REASONING_AGENT", "DIRECT_ANSWER"]
        if route not in valid_routes:
            logger.warning(f"Invalid route '{route}' — defaulting to RAG_AGENT")
            route = "RAG_AGENT"

        logger.info(f"Query routed to: {route}")
        return route

    except Exception as e:
        logger.error(f"Router failed: {e}")
        return "RAG_AGENT"

# Test router
print("=== Router Agent Test ===\n")

test_queries = [
    "What is RAG and how does it work?",
    "What is the latest news about OpenAI today?",
    "Calculate the fibonacci sequence up to 100",
    "Compare the pros and cons of RAG vs fine-tuning in detail",
    "Hello, how are you?",
    "How does FAISS indexing work?",
    "What is 2 + 2?",
]

for q in test_queries:
    route = route_query(q)
    print(f"Query : {q[:60]}")
    print(f"Route : {route}")
    print("-" * 50)

print("\n✅ Router Agent ready")

=== Router Agent Test ===

2026-03-17 16:57:53,021 | INFO | __main__ | Input guardrail passed: What is RAG and how does it work?...
2026-03-17 16:57:53,137 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 16:57:53,137 | INFO | __main__ | Query routed to: RAG_AGENT
Query : What is RAG and how does it work?
Route : RAG_AGENT
--------------------------------------------------
2026-03-17 16:57:53,137 | INFO | __main__ | Input guardrail passed: What is the latest news about OpenAI today?...
2026-03-17 16:57:53,233 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 16:57:53,233 | INFO | __main__ | Query routed to: WEB_SEARCH
Query : What is the latest news about OpenAI today?
Route : WEB_SEARCH
--------------------------------------------------
2026-03-17 16:57:53,233 | INFO | __main__ | Input guardrail passed: Calculate the fibonacci sequence up to 100...
2026-03-17 

In [24]:
REASONING_PROMPT = ChatPromptTemplate.from_template("""
You are an expert reasoning agent. Break down complex questions into steps
and reason through them carefully before giving a final answer.

Use this format:
STEP 1: [First reasoning step]
STEP 2: [Second reasoning step]
STEP 3: [Continue as needed]
CONCLUSION: [Final answer based on reasoning]

Context from knowledge base:
{context}

Web search results (if available):
{web_results}

Question: {question}

Reasoning:
""")

def reasoning_agent(
    query: str,
    use_rag: bool = True,
    use_web: bool = False
) -> dict:
    """
    Reasoning agent that combines RAG + web search for complex queries.
    
    Args:
        query: User query
        use_rag: Whether to use RAG retrieval
        use_web: Whether to use web search
    
    Returns:
        Dict with reasoning steps and final answer
    """
    try:
        logger.info(f"Reasoning agent processing: {query[:50]}...")

        # Retrieve context
        context = ""
        if use_rag:
            docs = base_retriever.invoke(query)
            context = format_docs(docs)
            logger.info(f"Retrieved {len(docs)} docs for reasoning")

        # Web search
        web_results = ""
        if use_web:
            web_results = web_search(query, max_results=2)
            logger.info("Web search completed for reasoning")

        # Run reasoning chain
        reasoning_chain = REASONING_PROMPT | llm | StrOutputParser()
        response = reasoning_chain.invoke({
            "context"    : context or "No context available",
            "web_results": web_results or "No web results",
            "question"   : query
        })

        # Parse steps from response
        lines = response.strip().split("\n")
        steps = [l for l in lines if l.strip().startswith("STEP")]
        conclusion = next((l for l in lines if "CONCLUSION" in l), "")

        return {
            "query"      : query,
            "steps"      : steps,
            "conclusion" : conclusion,
            "full_answer": response,
            "used_rag"   : use_rag,
            "used_web"   : use_web,
            "success"    : True
        }

    except Exception as e:
        logger.error(f"Reasoning agent failed: {e}")
        return {
            "query"      : query,
            "steps"      : [],
            "conclusion" : "",
            "full_answer": f"Reasoning failed: {str(e)}",
            "success"    : False
        }

# Test reasoning agent
print("=== Reasoning Agent Test ===\n")

complex_query = "What are the tradeoffs between using FAISS with reranking vs without reranking in a production RAG system?"

result = reasoning_agent(complex_query, use_rag=True, use_web=False)

print(f"Query: {result['query']}")
print(f"\nReasoning Steps ({len(result['steps'])}):")
for step in result["steps"]:
    print(f"  {step}")
print(f"\nConclusion: {result['conclusion']}")
print(f"\nSuccess: {result['success']}")

print("\n✅ Reasoning Agent ready")

=== Reasoning Agent Test ===

2026-03-17 16:58:18,614 | INFO | __main__ | Reasoning agent processing: What are the tradeoffs between using FAISS with re...
2026-03-17 16:58:18,679 | INFO | __main__ | Retrieved 4 docs for reasoning
2026-03-17 16:58:20,563 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
Query: What are the tradeoffs between using FAISS with reranking vs without reranking in a production RAG system?

Reasoning Steps (6):
  STEP 1: Understanding the components involved - FAISS is a library for efficient similarity search and clustering of dense vectors, often used in RAG (Retrieval-Augmented Generation) systems as a vector store. Reranking is a technique used to improve retrieval quality by scoring each retrieved document for relevance to the query after initial retrieval.
  STEP 2: Considering the purpose of reranking in a RAG system - Reranking aims to improve the accuracy of retrieved documents by reassessing their r

In [25]:
RAG_AGENT_PROMPT = ChatPromptTemplate.from_template("""
You are an expert RAG agent. Use the retrieved context to answer questions
accurately and concisely. Always cite your sources.

If the context doesn't contain the answer, say:
"I don't have enough information in my knowledge base to answer this."

Retrieved Context:
{context}

Conversation History:
{history}

Question: {question}

Answer:
""")

def rag_agent(
    query: str,
    memory_manager: MemoryManager = None,
    use_advanced: bool = True
) -> dict:
    """
    RAG agent with memory-aware context building.
    
    Args:
        query: User query
        memory_manager: Optional memory manager for conversation history
        use_advanced: Use advanced retriever with compression
    
    Returns:
        Dict with answer and metadata
    """
    try:
        logger.info(f"RAG agent processing: {query[:50]}...")

        # Choose retriever
        retriever = advanced_retriever if use_advanced else base_retriever

        # Retrieve docs
        docs = retriever.invoke(query)
        context = format_docs(docs)

        # Get conversation history
        history = ""
        if memory_manager:
            history = memory_manager.build_context(limit=4)

        # Build and run chain
        rag_agent_chain = RAG_AGENT_PROMPT | llm | StrOutputParser()
        answer = rag_agent_chain.invoke({
            "context" : context,
            "history" : history or "No previous conversation",
            "question": query
        })

        # Save to memory
        if memory_manager:
            memory_manager.remember("user", query)
            memory_manager.remember("assistant", answer[:500])

        # Validate output
        is_valid, reason, clean_answer = check_output_guardrails(answer)

        logger.info(f"RAG agent completed successfully")
        return {
            "query"        : query,
            "answer"       : clean_answer,
            "sources"      : [d.metadata.get("source") for d in docs],
            "num_docs"     : len(docs),
            "output_valid" : is_valid,
            "success"      : True
        }

    except Exception as e:
        logger.error(f"RAG agent failed: {e}")
        return {
            "query"  : query,
            "answer" : f"RAG agent failed: {str(e)}",
            "success": False
        }

# Test RAG agent
print("=== RAG Agent Test ===\n")

# Create a memory manager for this session
rag_session_manager = MemoryManager(memory_conn, "rag_agent_session_001")

test_queries = [
    "What is FAISS and why is it important for RAG?",
    "How does LangGraph manage state across nodes?",
    "What makes Groq different from other LLM providers?",
]

for q in test_queries:
    result = rag_agent(q, memory_manager=rag_session_manager)
    print(f"Query  : {q}")
    print(f"Answer : {result['answer'][:300]}...")
    print(f"Sources: {result['sources']}")
    print(f"Docs   : {result['num_docs']}")
    print("-" * 60)

print("\n✅ RAG Agent ready")

=== RAG Agent Test ===

2026-03-17 16:58:55,247 | INFO | __main__ | MemoryManager initialized for session: rag_agent_session_001
2026-03-17 16:58:55,249 | INFO | __main__ | RAG agent processing: What is FAISS and why is it important for RAG?...
2026-03-17 16:58:55,602 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 16:58:55,697 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 16:58:55,919 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 16:58:56,011 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 16:58:56,423 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 16:58:56,440 | INFO | __main__ | Output guardrail passed
2026-03-17 16:58:56,440 | INFO | __main__ | RAG agent com

In [26]:
from typing import TypedDict, Annotated, List, Optional
from langgraph.graph import StateGraph, END
import operator

class AgentState(TypedDict):
    """
    Shared state object passed between all LangGraph nodes.
    Every node reads from and writes to this state.
    """
    # Input
    query          : str
    session_id     : str

    # Routing
    route          : str
    is_safe        : bool
    safety_reason  : str

    # Retrieval
    retrieved_docs : List[str]
    context        : str

    # Web search
    web_results    : str

    # Code execution
    code_output    : str

    # Agent outputs
    reasoning      : str
    answer         : str

    # Evaluation
    eval_score     : float
    eval_feedback  : str
    needs_retry    : bool

    # Metadata
    retry_count    : int
    sources        : List[str]
    error          : Optional[str]
    success        : bool

def create_initial_state(query: str, session_id: str) -> AgentState:
    """Create a fresh initial state for a new query."""
    return AgentState(
        query          = query,
        session_id     = session_id,
        route          = "",
        is_safe        = True,
        safety_reason  = "",
        retrieved_docs = [],
        context        = "",
        web_results    = "",
        code_output    = "",
        reasoning      = "",
        answer         = "",
        eval_score     = 0.0,
        eval_feedback  = "",
        needs_retry    = False,
        retry_count    = 0,
        sources        = [],
        error          = None,
        success        = False
    )

# Test state creation
print("=== LangGraph State Test ===\n")

state = create_initial_state(
    query      = "What is RAG?",
    session_id = "graph_test_001"
)

print("Initial State:")
for key, value in state.items():
    print(f"  {key:20} : {value}")

print("\n✅ LangGraph State ready")

=== LangGraph State Test ===

Initial State:
  query                : What is RAG?
  session_id           : graph_test_001
  route                : 
  is_safe              : True
  safety_reason        : 
  retrieved_docs       : []
  context              : 
  web_results          : 
  code_output          : 
  reasoning            : 
  answer               : 
  eval_score           : 0.0
  eval_feedback        : 
  needs_retry          : False
  retry_count          : 0
  sources              : []
  error                : None
  success              : False

✅ LangGraph State ready


In [27]:
def node_input_guard(state: AgentState) -> AgentState:
    """
    Node 1: Validate and sanitize user input.
    Blocks unsafe queries before processing.
    """
    logger.info(f"[NODE] input_guard | query: {state['query'][:50]}")
    
    is_safe, reason, clean_query = check_input_guardrails(state["query"])
    
    if not is_safe:
        return {
            **state,
            "is_safe"      : False,
            "safety_reason": reason,
            "answer"       : f"Query blocked: {reason}",
            "success"      : False
        }
    
    return {
        **state,
        "query"       : clean_query,
        "is_safe"     : True,
        "safety_reason": "Input is safe"
    }


def node_router(state: AgentState) -> AgentState:
    """
    Node 2: Route query to appropriate agent.
    """
    logger.info(f"[NODE] router | query: {state['query'][:50]}")
    
    route = route_query(state["query"])
    
    return {
        **state,
        "route": route
    }


def node_retriever(state: AgentState) -> AgentState:
    """
    Node 3: Retrieve relevant documents from vector store.
    """
    logger.info(f"[NODE] retriever | query: {state['query'][:50]}")
    
    try:
        docs = base_retriever.invoke(state["query"])
        context = format_docs(docs)
        sources = [d.metadata.get("source", "unknown") for d in docs]
        
        return {
            **state,
            "retrieved_docs": [d.page_content for d in docs],
            "context"       : context,
            "sources"       : sources
        }
    except Exception as e:
        logger.error(f"Retriever node failed: {e}")
        return {**state, "error": str(e), "context": ""}


def node_web_search(state: AgentState) -> AgentState:
    """
    Node 4: Perform web search for current information.
    """
    logger.info(f"[NODE] web_search | query: {state['query'][:50]}")
    
    try:
        results = web_search(state["query"], max_results=3)
        return {**state, "web_results": results}
    except Exception as e:
        logger.error(f"Web search node failed: {e}")
        return {**state, "web_results": "", "error": str(e)}


def node_python_executor(state: AgentState) -> AgentState:
    """
    Node 5: Generate and execute Python code for computation queries.
    """
    logger.info(f"[NODE] python_executor | query: {state['query'][:50]}")
    
    try:
        # Ask LLM to generate code for the query
        code_prompt = f"""
Write Python code to answer this question.
Return ONLY executable Python code, no explanation.
Print the final answer using print().

Question: {state['query']}

Code:
"""
        code_response = llm.invoke(code_prompt).content
        
        # Extract code block if wrapped in markdown
        if "```python" in code_response:
            code = code_response.split("```python")[1].split("```")[0].strip()
        elif "```" in code_response:
            code = code_response.split("```")[1].split("```")[0].strip()
        else:
            code = code_response.strip()
        
        result = execute_python(code)
        output = result["stdout"] if result["success"] else result["error"]
        
        return {**state, "code_output": output}
    except Exception as e:
        logger.error(f"Python executor node failed: {e}")
        return {**state, "code_output": f"Execution failed: {str(e)}"}


def node_reasoning(state: AgentState) -> AgentState:
    """
    Node 6: Apply multi-step reasoning to complex queries.
    """
    logger.info(f"[NODE] reasoning | query: {state['query'][:50]}")
    
    result = reasoning_agent(
        state["query"],
        use_rag=True,
        use_web=False
    )
    
    return {
        **state,
        "reasoning": result.get("full_answer", ""),
        "answer"   : result.get("full_answer", "")
    }


def node_generator(state: AgentState) -> AgentState:
    """
    Node 7: Generate final answer using all gathered context.
    """
    logger.info(f"[NODE] generator | route: {state['route']}")
    
    try:
        # Build combined context based on route
        context_parts = []
        
        if state.get("context"):
            context_parts.append(f"Knowledge Base:\n{state['context']}")
        if state.get("web_results"):
            context_parts.append(f"Web Results:\n{state['web_results']}")
        if state.get("code_output"):
            context_parts.append(f"Code Output:\n{state['code_output']}")
        if state.get("reasoning"):
            context_parts.append(f"Reasoning:\n{state['reasoning']}")
        
        combined_context = "\n\n".join(context_parts) or "No context available"
        
        GENERATOR_PROMPT = ChatPromptTemplate.from_template("""
You are an expert AI assistant. Generate a clear, accurate, and helpful answer.
Use the provided context and cite sources when available.

Context:
{context}

Question: {question}

Answer:
""")
        
        gen_chain = GENERATOR_PROMPT | llm | StrOutputParser()
        answer = gen_chain.invoke({
            "context" : combined_context,
            "question": state["query"]
        })
        
        is_valid, reason, clean_answer = check_output_guardrails(answer)
        
        return {
            **state,
            "answer" : clean_answer,
            "success": True
        }
    
    except Exception as e:
        logger.error(f"Generator node failed: {e}")
        return {
            **state,
            "answer" : f"Generation failed: {str(e)}",
            "success": False,
            "error"  : str(e)
        }


def node_output_guard(state: AgentState) -> AgentState:
    """
    Node 8: Validate final output before returning to user.
    """
    logger.info(f"[NODE] output_guard")
    
    is_valid, reason, clean_answer = check_output_guardrails(state["answer"])
    
    return {
        **state,
        "answer" : clean_answer if is_valid else "Unable to generate a valid response.",
        "success": is_valid
    }


# Test individual nodes
print("=== LangGraph Nodes Test ===\n")

test_state = create_initial_state("What is FAISS?", "node_test_001")

# Run through nodes sequentially
test_state = node_input_guard(test_state)
print(f"After input_guard : is_safe={test_state['is_safe']}")

test_state = node_router(test_state)
print(f"After router      : route={test_state['route']}")

test_state = node_retriever(test_state)
print(f"After retriever   : docs={len(test_state['retrieved_docs'])}, sources={test_state['sources']}")

test_state = node_generator(test_state)
print(f"After generator   : answer={test_state['answer'][:100]}...")

test_state = node_output_guard(test_state)
print(f"After output_guard: success={test_state['success']}")

print("\n✅ All LangGraph Nodes ready")

=== LangGraph Nodes Test ===

2026-03-17 17:01:02,460 | INFO | __main__ | [NODE] input_guard | query: What is FAISS?
2026-03-17 17:01:02,462 | INFO | __main__ | Input guardrail passed: What is FAISS?...
After input_guard : is_safe=True
2026-03-17 17:01:02,464 | INFO | __main__ | [NODE] router | query: What is FAISS?
2026-03-17 17:01:02,464 | INFO | __main__ | Input guardrail passed: What is FAISS?...
2026-03-17 17:01:02,590 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 17:01:02,600 | INFO | __main__ | Query routed to: RAG_AGENT
After router      : route=RAG_AGENT
2026-03-17 17:01:02,604 | INFO | __main__ | [NODE] retriever | query: What is FAISS?
After retriever   : docs=4, sources=['faiss_overview.txt', 'faiss_overview.txt', 'groq_overview.txt', 'rag_overview.txt']
2026-03-17 17:01:02,645 | INFO | __main__ | [NODE] generator | route: RAG_AGENT
2026-03-17 17:01:02,996 | INFO | httpx | HTTP Request: POST https://api.groq

In [29]:
from langgraph.graph import StateGraph, END

def should_continue_after_guard(state: AgentState) -> str:
    """Edge: After input guard — continue or end."""
    if not state["is_safe"]:
        logger.info("Query blocked — ending graph")
        return "blocked"
    return "continue"

def route_after_router(state: AgentState) -> str:
    """Edge: After router — direct to correct node."""
    route = state["route"]
    logger.info(f"Routing to: {route}")
    
    route_map = {
        "RAG_AGENT"      : "retriever_node",
        "WEB_SEARCH"     : "web_search_node",
        "PYTHON_AGENT"   : "python_node",
        "REASONING_AGENT": "reasoning_node",
        "DIRECT_ANSWER"  : "generator_node",
        "BLOCKED"        : "output_guard_node",
    }
    return route_map.get(route, "retriever_node")

def should_retry(state: AgentState) -> str:
    """Edge: After evaluation — retry or end."""
    if state.get("needs_retry") and state.get("retry_count", 0) < 2:
        logger.info(f"Retrying — attempt {state['retry_count'] + 1}")
        return "retry"
    return "end"

# Build the graph
def build_workflow() -> StateGraph:
    """Build and compile the full agentic workflow graph."""
    
    workflow = StateGraph(AgentState)
    
    # Add all nodes — all names end with _node to avoid state key conflicts
    workflow.add_node("input_guard_node"  , node_input_guard)
    workflow.add_node("router_node"       , node_router)
    workflow.add_node("retriever_node"    , node_retriever)
    workflow.add_node("web_search_node"   , node_web_search)
    workflow.add_node("python_node"       , node_python_executor)
    workflow.add_node("reasoning_node"    , node_reasoning)
    workflow.add_node("generator_node"    , node_generator)
    workflow.add_node("output_guard_node" , node_output_guard)
    
    # Entry point
    workflow.set_entry_point("input_guard_node")
    
    # Conditional edge after input guard
    workflow.add_conditional_edges(
        "input_guard_node",
        should_continue_after_guard,
        {
            "continue": "router_node",
            "blocked" : "output_guard_node"
        }
    )
    
    # Conditional edge after router
    workflow.add_conditional_edges(
        "router_node",
        route_after_router,
        {
            "retriever_node"  : "retriever_node",
            "web_search_node" : "web_search_node",
            "python_node"     : "python_node",
            "reasoning_node"  : "reasoning_node",
            "generator_node"  : "generator_node",
            "output_guard_node": "output_guard_node"
        }
    )
    
    # All processing nodes flow to generator
    workflow.add_edge("retriever_node"  , "generator_node")
    workflow.add_edge("web_search_node" , "generator_node")
    workflow.add_edge("python_node"     , "generator_node")
    workflow.add_edge("reasoning_node"  , "output_guard_node")
    
    # Generator to output guard
    workflow.add_edge("generator_node", "output_guard_node")
    
    # Output guard to END
    workflow.add_edge("output_guard_node", END)
    
    return workflow.compile()

# Compile workflow
app = build_workflow()
logger.info("LangGraph workflow compiled successfully")

# Test workflow with multiple queries
print("=== LangGraph Workflow Test ===\n")

test_cases = [
    ("What is RAG?",                              "workflow_test_001"),
    ("Calculate sum of squares from 1 to 10",    "workflow_test_002"),
    ("How does FAISS similarity search work?",    "workflow_test_003"),
]

for query, session in test_cases:
    print(f"Query  : {query}")
    
    initial_state = create_initial_state(query, session)
    final_state   = app.invoke(initial_state)
    
    print(f"Route  : {final_state['route']}")
    print(f"Answer : {final_state['answer'][:200]}...")
    print(f"Sources: {final_state['sources']}")
    print(f"Success: {final_state['success']}")
    print("-" * 60)

print("\n✅ LangGraph Workflow ready")

2026-03-17 17:05:52,891 | INFO | __main__ | LangGraph workflow compiled successfully
=== LangGraph Workflow Test ===

Query  : What is RAG?
2026-03-17 17:05:52,891 | INFO | __main__ | [NODE] input_guard | query: What is RAG?
2026-03-17 17:05:52,891 | INFO | __main__ | Input guardrail passed: What is RAG?...
2026-03-17 17:05:52,903 | INFO | __main__ | [NODE] router | query: What is RAG?
2026-03-17 17:05:52,904 | INFO | __main__ | Input guardrail passed: What is RAG?...
2026-03-17 17:05:53,020 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 17:05:53,023 | INFO | __main__ | Query routed to: DIRECT_ANSWER
2026-03-17 17:05:53,023 | INFO | __main__ | Routing to: DIRECT_ANSWER
2026-03-17 17:05:53,028 | INFO | __main__ | [NODE] generator | route: DIRECT_ANSWER
2026-03-17 17:05:54,108 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 17:05:54,108 | INFO | __main__ | Ou

In [30]:
EVAL_PROMPT = ChatPromptTemplate.from_template("""
You are an expert AI evaluator. Score the following answer on these criteria:

1. RELEVANCE  : Does the answer address the question? (0-10)
2. ACCURACY   : Is the answer factually correct? (0-10)
3. COMPLETENESS: Does it cover all aspects? (0-10)
4. CLARITY    : Is it clear and well-structured? (0-10)

Respond in EXACTLY this format:
RELEVANCE: <score>
ACCURACY: <score>
COMPLETENESS: <score>
CLARITY: <score>
OVERALL: <average score>
FEEDBACK: <one sentence of feedback>
NEEDS_RETRY: <YES or NO>

Question: {question}
Answer: {answer}

Evaluation:
""")

def evaluate_response(question: str, answer: str) -> dict:
    """
    LLM-as-a-judge evaluation of a question-answer pair.
    
    Args:
        question: Original user question
        answer: Generated answer to evaluate
    
    Returns:
        Dict with scores and feedback
    """
    try:
        logger.info(f"Evaluating response for: {question[:50]}...")
        
        eval_chain = EVAL_PROMPT | llm | StrOutputParser()
        eval_output = eval_chain.invoke({
            "question": question,
            "answer"  : answer
        })
        
        # Parse scores from output
        scores = {}
        needs_retry = False
        feedback = ""
        
        for line in eval_output.strip().split("\n"):
            line = line.strip()
            if line.startswith("RELEVANCE:"):
                scores["relevance"] = float(line.split(":")[1].strip())
            elif line.startswith("ACCURACY:"):
                scores["accuracy"] = float(line.split(":")[1].strip())
            elif line.startswith("COMPLETENESS:"):
                scores["completeness"] = float(line.split(":")[1].strip())
            elif line.startswith("CLARITY:"):
                scores["clarity"] = float(line.split(":")[1].strip())
            elif line.startswith("OVERALL:"):
                scores["overall"] = float(line.split(":")[1].strip())
            elif line.startswith("FEEDBACK:"):
                feedback = line.split(":", 1)[1].strip()
            elif line.startswith("NEEDS_RETRY:"):
                needs_retry = "YES" in line.upper()
        
        # Default overall if not parsed
        if "overall" not in scores and scores:
            scores["overall"] = sum(scores.values()) / len(scores)
        
        logger.info(f"Evaluation complete — overall: {scores.get('overall', 0):.1f}/10")
        
        return {
            "scores"      : scores,
            "feedback"    : feedback,
            "needs_retry" : needs_retry,
            "raw_output"  : eval_output,
            "success"     : True
        }
    
    except Exception as e:
        logger.error(f"Evaluation failed: {e}")
        return {
            "scores"     : {"overall": 5.0},
            "feedback"   : "Evaluation failed",
            "needs_retry": False,
            "success"    : False
        }

# Test evaluator
print("=== LLM Evaluation (Judge) Test ===\n")

eval_pairs = [
    (
        "What is RAG?",
        "RAG stands for Retrieval Augmented Generation. It enhances LLMs by retrieving relevant documents from a knowledge base before generating responses, reducing hallucinations."
    ),
    (
        "What is FAISS?",
        "I don't know."
    ),
    (
        "How does LangGraph work?",
        "LangGraph is a framework for building stateful agentic workflows using graph-based orchestration where nodes process data and edges define flow between steps."
    ),
]

for question, answer in eval_pairs:
    result = evaluate_response(question, answer)
    print(f"Question : {question}")
    print(f"Answer   : {answer[:80]}...")
    print(f"Scores   : {result['scores']}")
    print(f"Feedback : {result['feedback']}")
    print(f"Retry?   : {result['needs_retry']}")
    print("-" * 60)

print("\n✅ LLM Evaluation ready")

=== LLM Evaluation (Judge) Test ===

2026-03-17 17:06:05,959 | INFO | __main__ | Evaluating response for: What is RAG?...
2026-03-17 17:06:06,504 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 17:06:06,511 | INFO | __main__ | Evaluation complete — overall: 9.2/10
Question : What is RAG?
Answer   : RAG stands for Retrieval Augmented Generation. It enhances LLMs by retrieving re...
Scores   : {'relevance': 10.0, 'accuracy': 10.0, 'completeness': 8.0, 'clarity': 9.0, 'overall': 9.25}
Feedback : The answer is clear and accurate, but could be improved by providing more details about the benefits and applications of RAG.
Retry?   : False
------------------------------------------------------------
2026-03-17 17:06:06,512 | INFO | __main__ | Evaluating response for: What is FAISS?...
2026-03-17 17:06:06,909 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 17:06:06,

In [31]:
import time
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

# Retry decorator for LLM calls
@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type(Exception),
    reraise=True
)
def llm_with_retry(prompt: str) -> str:
    """LLM call with automatic retry on failure."""
    response = llm.invoke(prompt)
    return response.content

def pipeline_with_retry(
    query: str,
    session_id: str,
    max_retries: int = 2,
    min_eval_score: float = 6.0
) -> dict:
    """
    Run full pipeline with evaluation-based retry.
    Retries if evaluation score is below threshold.
    
    Args:
        query: User query
        session_id: Session identifier
        max_retries: Maximum retry attempts
        min_eval_score: Minimum acceptable eval score
    
    Returns:
        Dict with final answer and metadata
    """
    attempt = 0
    best_result = None
    best_score = 0.0
    
    while attempt <= max_retries:
        logger.info(f"Pipeline attempt {attempt + 1}/{max_retries + 1}")
        
        try:
            # Run graph
            state = create_initial_state(query, f"{session_id}_attempt_{attempt}")
            final_state = app.invoke(state)
            
            # Evaluate result
            eval_result = evaluate_response(query, final_state["answer"])
            score = eval_result["scores"].get("overall", 0.0)
            
            logger.info(f"Attempt {attempt + 1} score: {score:.1f}/10")
            
            # Track best result
            if score > best_score:
                best_score = score
                best_result = {
                    "answer"     : final_state["answer"],
                    "score"      : score,
                    "feedback"   : eval_result["feedback"],
                    "attempt"    : attempt + 1,
                    "sources"    : final_state["sources"],
                    "route"      : final_state["route"],
                    "success"    : True
                }
            
            # Check if score is acceptable
            if score >= min_eval_score:
                logger.info(f"Acceptable score {score:.1f} — stopping retries")
                break
            
            attempt += 1
            if attempt <= max_retries:
                logger.info(f"Score {score:.1f} below threshold {min_eval_score} — retrying...")
                time.sleep(1)
        
        except Exception as e:
            logger.error(f"Pipeline attempt {attempt + 1} failed: {e}")
            attempt += 1
    
    if best_result is None:
        return {
            "answer" : "Pipeline failed after all retries",
            "score"  : 0.0,
            "success": False
        }
    
    best_result["total_attempts"] = attempt
    return best_result

# Test retry mechanism
print("=== Retry Mechanism Test ===\n")

result = pipeline_with_retry(
    query         = "Explain how reranking improves RAG pipeline quality",
    session_id    = "retry_test_001",
    max_retries   = 2,
    min_eval_score= 6.0
)

print(f"Final Answer  : {result['answer'][:300]}...")
print(f"Score         : {result['score']:.1f}/10")
print(f"Feedback      : {result['feedback']}")
print(f"Attempts      : {result['total_attempts']}")
print(f"Route         : {result.get('route', 'N/A')}")
print(f"Success       : {result['success']}")

print("\n✅ Retry Mechanism ready")

=== Retry Mechanism Test ===

2026-03-17 17:06:38,993 | INFO | __main__ | Pipeline attempt 1/3
2026-03-17 17:06:38,997 | INFO | __main__ | [NODE] input_guard | query: Explain how reranking improves RAG pipeline qualit
2026-03-17 17:06:38,997 | INFO | __main__ | Input guardrail passed: Explain how reranking improves RAG pipeline qualit...
2026-03-17 17:06:38,998 | INFO | __main__ | [NODE] router | query: Explain how reranking improves RAG pipeline qualit
2026-03-17 17:06:38,998 | INFO | __main__ | Input guardrail passed: Explain how reranking improves RAG pipeline qualit...
2026-03-17 17:06:39,095 | INFO | httpx | HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-17 17:06:39,107 | INFO | __main__ | Query routed to: REASONING_AGENT
2026-03-17 17:06:39,108 | INFO | __main__ | Routing to: REASONING_AGENT
2026-03-17 17:06:39,108 | INFO | __main__ | [NODE] reasoning | query: Explain how reranking improves RAG pipeline qualit
2026-03-17 17:06:39,111 

In [32]:
import hashlib
import json
import time

class ResponseCache:
    """
    Simple in-memory + SQLite backed response cache.
    Avoids redundant LLM calls for identical queries.
    """
    
    def __init__(self, conn: sqlite3.Connection, ttl: int = 3600):
        """
        Initialize cache.
        
        Args:
            conn: SQLite connection
            ttl: Time to live in seconds (default 1 hour)
        """
        self.conn  = conn
        self.ttl   = ttl
        self._init_cache_table()
        logger.info(f"ResponseCache initialized with TTL={ttl}s")
    
    def _init_cache_table(self):
        """Create cache table if not exists."""
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS response_cache (
                cache_key  TEXT PRIMARY KEY,
                query      TEXT NOT NULL,
                response   TEXT NOT NULL,
                score      REAL,
                created_at REAL NOT NULL
            )
        """)
        self.conn.commit()
    
    def _make_key(self, query: str) -> str:
        """Generate cache key from query."""
        return hashlib.md5(query.lower().strip().encode()).hexdigest()
    
    def get(self, query: str) -> dict:
        """
        Retrieve cached response if exists and not expired.
        
        Args:
            query: User query
        
        Returns:
            Cached response dict or None
        """
        key = self._make_key(query)
        cursor = self.conn.execute(
            "SELECT query, response, score, created_at FROM response_cache WHERE cache_key=?",
            (key,)
        )
        row = cursor.fetchone()
        
        if not row:
            logger.info(f"Cache MISS: {query[:50]}")
            return None
        
        # Check TTL
        age = time.time() - row[3]
        if age > self.ttl:
            logger.info(f"Cache EXPIRED: {query[:50]} (age={age:.0f}s)")
            self.delete(query)
            return None
        
        logger.info(f"Cache HIT: {query[:50]} (age={age:.0f}s)")
        return {
            "query"    : row[0],
            "response" : row[1],
            "score"    : row[2],
            "age_secs" : age,
            "from_cache": True
        }
    
    def set(self, query: str, response: str, score: float = 0.0):
        """
        Store response in cache.
        
        Args:
            query: User query
            response: Generated response
            score: Evaluation score
        """
        key = self._make_key(query)
        self.conn.execute(
            "INSERT OR REPLACE INTO response_cache (cache_key, query, response, score, created_at) VALUES (?, ?, ?, ?, ?)",
            (key, query, response, score, time.time())
        )
        self.conn.commit()
        logger.info(f"Cache SET: {query[:50]}")
    
    def delete(self, query: str):
        """Remove a specific entry from cache."""
        key = self._make_key(query)
        self.conn.execute("DELETE FROM response_cache WHERE cache_key=?", (key,))
        self.conn.commit()
    
    def clear_expired(self):
        """Remove all expired cache entries."""
        cutoff = time.time() - self.ttl
        self.conn.execute("DELETE FROM response_cache WHERE created_at < ?", (cutoff,))
        self.conn.commit()
        logger.info("Cleared expired cache entries")
    
    def stats(self) -> dict:
        """Get cache statistics."""
        cursor = self.conn.execute("SELECT COUNT(*), AVG(score) FROM response_cache")
        row = cursor.fetchone()
        return {
            "total_entries": row[0],
            "avg_score"    : round(row[1] or 0, 2)
        }

# Test cache
print("=== Response Cache Test ===\n")

cache = ResponseCache(memory_conn, ttl=3600)

# Miss — not in cache yet
query1 = "What is RAG?"
result = cache.get(query1)
print(f"First lookup (should be MISS): {result}")

# Set cache
cache.set(query1, "RAG is Retrieval Augmented Generation...", score=8.5)

# Hit — now in cache
result = cache.get(query1)
print(f"\nSecond lookup (should be HIT):")
print(f"  From cache : {result['from_cache']}")
print(f"  Response   : {result['response'][:60]}...")
print(f"  Score      : {result['score']}")
print(f"  Age        : {result['age_secs']:.2f}s")

# Test with multiple entries
cache.set("What is FAISS?", "FAISS is a vector similarity search library...", score=9.0)
cache.set("How does LangGraph work?", "LangGraph uses graph-based orchestration...", score=7.5)

stats = cache.stats()
print(f"\nCache Stats: {stats}")

print("\n✅ Response Cache ready")

=== Response Cache Test ===

2026-03-17 17:06:58,347 | INFO | __main__ | ResponseCache initialized with TTL=3600s
2026-03-17 17:06:58,349 | INFO | __main__ | Cache MISS: What is RAG?
First lookup (should be MISS): None
2026-03-17 17:06:58,351 | INFO | __main__ | Cache SET: What is RAG?
2026-03-17 17:06:58,351 | INFO | __main__ | Cache HIT: What is RAG? (age=0s)

Second lookup (should be HIT):
  From cache : True
  Response   : RAG is Retrieval Augmented Generation......
  Score      : 8.5
  Age        : 0.00s
2026-03-17 17:06:58,356 | INFO | __main__ | Cache SET: What is FAISS?
2026-03-17 17:06:58,356 | INFO | __main__ | Cache SET: How does LangGraph work?

Cache Stats: {'total_entries': 3, 'avg_score': 8.33}

✅ Response Cache ready


In [33]:
import time
import uuid
from datetime import datetime
from dataclasses import dataclass, field
from typing import Dict, List

@dataclass
class Span:
    """Represents a single traced operation."""
    span_id   : str
    name      : str
    start_time: float
    end_time  : float  = 0.0
    duration  : float  = 0.0
    metadata  : Dict   = field(default_factory=dict)
    status    : str    = "running"
    error     : str    = ""

@dataclass
class Trace:
    """Represents a full request trace across all nodes."""
    trace_id  : str
    query     : str
    spans     : List[Span] = field(default_factory=list)
    start_time: float      = field(default_factory=time.time)
    end_time  : float      = 0.0
    total_time: float      = 0.0
    success   : bool       = False

class ObservabilityTracer:
    """
    Lightweight tracer for monitoring agentic pipeline execution.
    Tracks spans, latency, and errors across all nodes.
    """
    
    def __init__(self):
        self.traces: Dict[str, Trace] = {}
        self.metrics = {
            "total_requests"  : 0,
            "successful"      : 0,
            "failed"          : 0,
            "avg_latency_ms"  : 0.0,
            "cache_hits"      : 0,
            "cache_misses"    : 0,
            "total_latency_ms": 0.0,
        }
        logger.info("ObservabilityTracer initialized")
    
    def start_trace(self, query: str) -> str:
        """Start a new trace for a request."""
        trace_id = str(uuid.uuid4())[:8]
        self.traces[trace_id] = Trace(
            trace_id = trace_id,
            query    = query
        )
        self.metrics["total_requests"] += 1
        logger.info(f"[TRACE START] {trace_id} | {query[:50]}")
        return trace_id
    
    def start_span(self, trace_id: str, name: str, metadata: dict = None) -> str:
        """Start a new span within a trace."""
        span_id = str(uuid.uuid4())[:6]
        span = Span(
            span_id    = span_id,
            name       = name,
            start_time = time.time(),
            metadata   = metadata or {}
        )
        if trace_id in self.traces:
            self.traces[trace_id].spans.append(span)
        logger.info(f"  [SPAN START] {name} ({span_id})")
        return span_id
    
    def end_span(self, trace_id: str, span_id: str, status: str = "ok", error: str = ""):
        """End a span and record duration."""
        if trace_id not in self.traces:
            return
        for span in self.traces[trace_id].spans:
            if span.span_id == span_id:
                span.end_time = time.time()
                span.duration = (span.end_time - span.start_time) * 1000
                span.status   = status
                span.error    = error
                logger.info(f"  [SPAN END] {span.name} | {span.duration:.1f}ms | {status}")
                break
    
    def end_trace(self, trace_id: str, success: bool = True):
        """End a trace and compute total duration."""
        if trace_id not in self.traces:
            return
        trace = self.traces[trace_id]
        trace.end_time  = time.time()
        trace.total_time = (trace.end_time - trace.start_time) * 1000
        trace.success   = success
        
        # Update metrics
        if success:
            self.metrics["successful"] += 1
        else:
            self.metrics["failed"] += 1
        
        self.metrics["total_latency_ms"] += trace.total_time
        self.metrics["avg_latency_ms"] = (
            self.metrics["total_latency_ms"] / self.metrics["total_requests"]
        )
        
        logger.info(f"[TRACE END] {trace_id} | {trace.total_time:.1f}ms | success={success}")
    
    def get_trace_summary(self, trace_id: str) -> dict:
        """Get summary of a trace."""
        if trace_id not in self.traces:
            return {}
        trace = self.traces[trace_id]
        return {
            "trace_id"  : trace.trace_id,
            "query"     : trace.query[:60],
            "total_ms"  : round(trace.total_time, 1),
            "success"   : trace.success,
            "num_spans" : len(trace.spans),
            "spans"     : [
                {
                    "name"    : s.name,
                    "duration": round(s.duration, 1),
                    "status"  : s.status
                }
                for s in trace.spans
            ]
        }
    
    def get_metrics(self) -> dict:
        """Get overall system metrics."""
        return {k: round(v, 2) if isinstance(v, float) else v 
                for k, v in self.metrics.items()}

# Test observability
print("=== Observability Tracer Test ===\n")

tracer = ObservabilityTracer()

# Simulate a traced pipeline run
trace_id = tracer.start_trace("What is RAG and how does it work?")

span1 = tracer.start_span(trace_id, "input_guard")
time.sleep(0.01)
tracer.end_span(trace_id, span1, "ok")

span2 = tracer.start_span(trace_id, "router")
time.sleep(0.02)
tracer.end_span(trace_id, span2, "ok")

span3 = tracer.start_span(trace_id, "retriever", {"num_docs": 4})
time.sleep(0.05)
tracer.end_span(trace_id, span3, "ok")

span4 = tracer.start_span(trace_id, "generator")
time.sleep(0.03)
tracer.end_span(trace_id, span4, "ok")

span5 = tracer.start_span(trace_id, "output_guard")
time.sleep(0.01)
tracer.end_span(trace_id, span5, "ok")

tracer.end_trace(trace_id, success=True)

# Print trace summary
summary = tracer.get_trace_summary(trace_id)
print("Trace Summary:")
print(f"  Trace ID  : {summary['trace_id']}")
print(f"  Query     : {summary['query']}")
print(f"  Total     : {summary['total_ms']}ms")
print(f"  Success   : {summary['success']}")
print(f"  Spans     : {summary['num_spans']}")
print(f"\n  Span Breakdown:")
for span in summary["spans"]:
    bar = "█" * int(span["duration"] / 5)
    print(f"    {span['name']:20} {span['duration']:6.1f}ms {bar}")

metrics = tracer.get_metrics()
print(f"\nSystem Metrics: {metrics}")

print("\n✅ Observability Tracer ready")

=== Observability Tracer Test ===

2026-03-17 17:07:08,397 | INFO | __main__ | ObservabilityTracer initialized
2026-03-17 17:07:08,405 | INFO | __main__ | [TRACE START] e85df724 | What is RAG and how does it work?
2026-03-17 17:07:08,405 | INFO | __main__ |   [SPAN START] input_guard (bfd4bf)
2026-03-17 17:07:08,418 | INFO | __main__ |   [SPAN END] input_guard | 12.2ms | ok
2026-03-17 17:07:08,418 | INFO | __main__ |   [SPAN START] router (cc5edd)
2026-03-17 17:07:08,440 | INFO | __main__ |   [SPAN END] router | 22.1ms | ok
2026-03-17 17:07:08,440 | INFO | __main__ |   [SPAN START] retriever (b1d1bc)
2026-03-17 17:07:08,493 | INFO | __main__ |   [SPAN END] retriever | 53.2ms | ok
2026-03-17 17:07:08,493 | INFO | __main__ |   [SPAN START] generator (dd231e)
2026-03-17 17:07:08,525 | INFO | __main__ |   [SPAN END] generator | 32.0ms | ok
2026-03-17 17:07:08,525 | INFO | __main__ |   [SPAN START] output_guard (055492)
2026-03-17 17:07:08,537 | INFO | __main__ |   [SPAN END] output_guard |

In [34]:
import time
from datetime import datetime

def run_full_pipeline(
    query      : str,
    session_id : str = None,
    use_cache  : bool = True,
    min_score  : float = 6.0,
    max_retries: int = 2
) -> dict:
    """
    Full end-to-end agentic RAG pipeline.
    
    Integrates:
    - Input/Output Guardrails
    - Router Agent
    - RAG (Advanced Retrieval + Reranking + Compression)
    - Web Search Tool
    - Python Executor Tool
    - Database Tool
    - Short + Long Term Memory
    - LangGraph Workflow
    - LLM Evaluation (Judge)
    - Retry Mechanism
    - Response Cache
    - Observability (Tracing)
    
    Args:
        query      : User query string
        session_id : Session identifier (auto-generated if None)
        use_cache  : Whether to use response cache
        min_score  : Minimum evaluation score before retry
        max_retries: Maximum retry attempts
    
    Returns:
        Dict with answer, metadata, scores, trace
    """
    
    # Generate session ID if not provided
    if not session_id:
        session_id = f"session_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    
    pipeline_start = time.time()
    
    # ── Step 1: Start Observability Trace ────────────────
    trace_id = tracer.start_trace(query)
    logger.info(f"{'='*60}")
    logger.info(f"PIPELINE START | session={session_id} | trace={trace_id}")
    logger.info(f"Query: {query}")
    logger.info(f"{'='*60}")
    
    # ── Step 2: Input Guardrails ──────────────────────────
    span_guard = tracer.start_span(trace_id, "input_guard")
    is_safe, reason, clean_query = check_input_guardrails(query)
    tracer.end_span(trace_id, span_guard, "ok" if is_safe else "blocked")
    
    if not is_safe:
        tracer.end_trace(trace_id, success=False)
        return {
            "query"    : query,
            "answer"   : f"Query blocked: {reason}",
            "success"  : False,
            "blocked"  : True,
            "reason"   : reason,
            "trace_id" : trace_id,
            "session_id": session_id
        }
    
    # ── Step 3: Check Cache ───────────────────────────────
    span_cache = tracer.start_span(trace_id, "cache_lookup")
    if use_cache:
        cached = cache.get(clean_query)
        if cached:
            tracer.end_span(trace_id, span_cache, "hit")
            tracer.end_trace(trace_id, success=True)
            tracer.metrics["cache_hits"] += 1
            logger.info(f"Cache HIT — returning cached response")
            return {
                "query"      : clean_query,
                "answer"     : cached["response"],
                "score"      : cached["score"],
                "from_cache" : True,
                "trace_id"   : trace_id,
                "session_id" : session_id,
                "success"    : True
            }
    tracer.end_span(trace_id, span_cache, "miss")
    tracer.metrics["cache_misses"] += 1
    
    # ── Step 4: Memory — Load Context ────────────────────
    span_memory = tracer.start_span(trace_id, "memory_load")
    mem_manager = MemoryManager(memory_conn, session_id)
    conversation_history = mem_manager.build_context(limit=4)
    user_profile = mem_manager.get_user_profile()
    tracer.end_span(trace_id, span_memory, "ok")
    logger.info(f"Memory loaded | history_len={len(conversation_history)}")
    
    # ── Step 5: Router ────────────────────────────────────
    span_router = tracer.start_span(trace_id, "router")
    route = route_query(clean_query)
    tracer.end_span(trace_id, span_router, "ok")
    logger.info(f"Routed to: {route}")
    
    # ── Step 6: LangGraph Workflow ────────────────────────
    span_graph = tracer.start_span(trace_id, "langgraph_workflow")
    
    best_answer  = ""
    best_score   = 0.0
    best_feedback = ""
    attempt      = 0
    
    while attempt <= max_retries:
        logger.info(f"Workflow attempt {attempt + 1}/{max_retries + 1}")
        
        try:
            # Run LangGraph
            initial_state = create_initial_state(
                clean_query,
                f"{session_id}_attempt_{attempt}"
            )
            
            span_nodes = tracer.start_span(trace_id, f"graph_attempt_{attempt+1}")
            final_state = app.invoke(initial_state)
            tracer.end_span(trace_id, span_nodes, "ok")
            
            current_answer = final_state.get("answer", "")
            sources        = final_state.get("sources", [])
            graph_route    = final_state.get("route", route)
            
            # ── Step 7: Evaluate Output ───────────────────
            span_eval = tracer.start_span(trace_id, f"evaluation_{attempt+1}")
            eval_result = evaluate_response(clean_query, current_answer)
            score       = eval_result["scores"].get("overall", 0.0)
            feedback    = eval_result["feedback"]
            tracer.end_span(trace_id, span_eval, "ok")
            
            logger.info(f"Attempt {attempt+1} score: {score:.1f}/10 | {feedback}")
            
            # Track best
            if score > best_score:
                best_score    = score
                best_answer   = current_answer
                best_feedback = feedback
                best_sources  = sources
                best_route    = graph_route
            
            # Accept if score is good enough
            if score >= min_score:
                logger.info(f"Score {score:.1f} >= threshold {min_score} — accepting")
                break
            
            attempt += 1
            if attempt <= max_retries:
                logger.warning(f"Score {score:.1f} below threshold — retrying...")
                time.sleep(1)
        
        except Exception as e:
            logger.error(f"Workflow attempt {attempt+1} failed: {e}")
            tracer.end_span(trace_id, span_graph, "error")
            attempt += 1
    
    tracer.end_span(trace_id, span_graph, "ok")
    
    # ── Step 8: Output Guardrails ─────────────────────────
    span_out = tracer.start_span(trace_id, "output_guard")
    is_valid, out_reason, clean_answer = check_output_guardrails(best_answer)
    tracer.end_span(trace_id, span_out, "ok" if is_valid else "invalid")
    
    # ── Step 9: Save to Memory ────────────────────────────
    span_mem_save = tracer.start_span(trace_id, "memory_save")
    mem_manager.remember("user", clean_query)
    mem_manager.remember("assistant", clean_answer[:500])
    if best_score >= 8.0:
        mem_manager.learn(
            f"qa_{session_id}",
            f"Q: {clean_query[:100]} A: {clean_answer[:200]}",
            category="qa_history"
        )
    tracer.end_span(trace_id, span_mem_save, "ok")
    
    # ── Step 10: Save to Cache ────────────────────────────
    span_cache_set = tracer.start_span(trace_id, "cache_save")
    if use_cache and best_score >= min_score:
        cache.set(clean_query, clean_answer, score=best_score)
    tracer.end_span(trace_id, span_cache_set, "ok")
    
    # ── Step 11: Save Tool Result to DB ──────────────────
    save_tool_result(
        tool_db,
        tool_name = "full_pipeline",
        query     = clean_query,
        result    = clean_answer[:500],
        success   = is_valid
    )
    
    # ── Step 12: End Trace ────────────────────────────────
    tracer.end_trace(trace_id, success=is_valid)
    
    total_time = (time.time() - pipeline_start) * 1000
    
    logger.info(f"{'='*60}")
    logger.info(f"PIPELINE END | time={total_time:.0f}ms | score={best_score:.1f} | success={is_valid}")
    logger.info(f"{'='*60}")
    
    return {
        "query"          : clean_query,
        "answer"         : clean_answer,
        "score"          : best_score,
        "feedback"       : best_feedback,
        "sources"        : best_sources if best_score > 0 else [],
        "route"          : best_route if best_score > 0 else route,
        "attempts"       : attempt,
        "from_cache"     : False,
        "output_valid"   : is_valid,
        "trace_id"       : trace_id,
        "session_id"     : session_id,
        "total_time_ms"  : round(total_time, 1),
        "success"        : is_valid
    }


# ── RUN FULL PIPELINE TESTS ──────────────────────────────
print("=" * 70)
print("       FULL AGENTIC RAG PIPELINE — END TO END TEST")
print("=" * 70)

test_queries = [
    {
        "query"     : "What is RAG and how does it reduce hallucinations?",
        "session_id": "full_test_001",
        "description": "RAG Knowledge Base Query"
    },
    {
        "query"     : "How does FAISS enable fast similarity search?",
        "session_id": "full_test_002",
        "description": "Vector DB Query"
    },
    {
        "query"     : "Calculate the sum of first 10 prime numbers",
        "session_id": "full_test_003",
        "description": "Python Computation Query"
    },
    {
        "query"     : "Compare RAG vs fine-tuning for production AI systems",
        "session_id": "full_test_004",
        "description": "Complex Reasoning Query"
    },
    {
        "query"     : "What is RAG and how does it reduce hallucinations?",
        "session_id": "full_test_005",
        "description": "Cache Hit Test (same as query 1)"
    },
]

results = []

for i, test in enumerate(test_queries):
    print(f"\n{'─'*70}")
    print(f"Test {i+1}/5 : {test['description']}")
    print(f"Query      : {test['query']}")
    print(f"{'─'*70}")
    
    result = run_full_pipeline(
        query      = test["query"],
        session_id = test["session_id"],
        use_cache  = True,
        min_score  = 6.0,
        max_retries= 1
    )
    
    results.append(result)
    
    print(f"Route      : {result.get('route', 'N/A')}")
    print(f"Answer     : {result['answer'][:250]}...")
    print(f"Score      : {result['score']:.1f}/10")
    print(f"Feedback   : {result.get('feedback', 'N/A')}")
    print(f"Sources    : {result.get('sources', [])}")
    print(f"From Cache : {result.get('from_cache', False)}")
    print(f"Attempts   : {result.get('attempts', 1)}")
    print(f"Time       : {result.get('total_time_ms', 0):.0f}ms")
    print(f"Success    : {result['success']}")

# ── FINAL SUMMARY ─────────────────────────────────────────
print(f"\n{'='*70}")
print("                    PIPELINE SUMMARY")
print(f"{'='*70}")

successful  = sum(1 for r in results if r["success"])
avg_score   = sum(r["score"] for r in results) / len(results)
avg_time    = sum(r.get("total_time_ms", 0) for r in results) / len(results)
cache_hits  = sum(1 for r in results if r.get("from_cache"))

print(f"Total Queries   : {len(results)}")
print(f"Successful      : {successful}/{len(results)}")
print(f"Avg Score       : {avg_score:.1f}/10")
print(f"Avg Latency     : {avg_time:.0f}ms")
print(f"Cache Hits      : {cache_hits}/{len(results)}")

print(f"\nSystem Metrics:")
metrics = tracer.get_metrics()
for k, v in metrics.items():
    print(f"  {k:25}: {v}")

print(f"\nCache Stats     : {cache.stats()}")

print(f"\n{'='*70}")
print("✅ FULL PIPELINE COMPLETE — ALL SYSTEMS OPERATIONAL")
print(f"{'='*70}")


       FULL AGENTIC RAG PIPELINE — END TO END TEST

──────────────────────────────────────────────────────────────────────
Test 1/5 : RAG Knowledge Base Query
Query      : What is RAG and how does it reduce hallucinations?
──────────────────────────────────────────────────────────────────────
2026-03-17 17:09:37,195 | INFO | __main__ | [TRACE START] 03db3a63 | What is RAG and how does it reduce hallucinations?
2026-03-17 17:09:37,195 | INFO | __main__ | ============================================================
2026-03-17 17:09:37,195 | INFO | __main__ | PIPELINE START | session=full_test_001 | trace=03db3a63
2026-03-17 17:09:37,195 | INFO | __main__ | Query: What is RAG and how does it reduce hallucinations?
2026-03-17 17:09:37,195 | INFO | __main__ | ============================================================
2026-03-17 17:09:37,195 | INFO | __main__ |   [SPAN START] input_guard (4f9967)
2026-03-17 17:09:37,195 | INFO | __main__ | Input guardrail passed: What is RAG and how does i